# Churn Prediction - Exploratory Data Analysis (EDA)

**Goal:** Understand customer churn patterns using data from:
- `customers` - Customer demographics & status
- `subscriptions` - Subscription details
- `subscription_events` - Plan changes (upgrades/downgrades)
- `support_tickets` - Customer support interactions
- `activity_metrics` - Daily usage & engagement metrics

**Target:** `customers.status` → `'inactive'` means churned

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries loaded successfully')

In [ ]:
# Load all datasets
customers = pd.read_csv('data/customers.csv')
subscriptions = pd.read_csv('data/subscriptions.csv')
sub_events = pd.read_csv('data/subscription_events.csv')
tickets = pd.read_csv('data/support_tickets.csv')
activity = pd.read_csv('data/activity_metrics.csv')

print(f'Customers: {customers.shape}')
print(f'Subscriptions: {subscriptions.shape}')
print(f'Subscription Events: {sub_events.shape}')
print(f'Support Tickets: {tickets.shape}')
print(f'Activity Metrics: {activity.shape}')

In [ ]:
customers.head()

In [ ]:
subscriptions.head()

In [ ]:
sub_events.head()

In [ ]:
tickets.head()

In [ ]:
activity.head()

---
## 1. Target Variable: Churn Distribution
Churn is defined as `status == 'inactive'`

In [ ]:
churn_counts = customers['status'].value_counts()
print('Churn Distribution:')
print(churn_counts)
print(f'\nChurn Rate: {churn_counts.get("inactive", 0)/len(customers)*100:.2f}%')

fig = make_subplots(rows=1, cols=2, subplot_titles=['Churn Count', 'Churn Percentage'])
fig.add_trace(go.Bar(x=churn_counts.index, y=churn_counts.values, 
                     marker_color=['#2ecc71', '#e74c3c'], text=churn_counts.values), row=1, col=1)
fig.add_trace(go.Pie(labels=churn_counts.index, values=churn_counts.values,
                     marker_colors=['#2ecc71', '#e74c3c'], hole=0.4), row=1, col=2)
fig.update_layout(title_text='Churn Distribution', height=400)
fig.show()

## 2. Customer Demographics Analysis

In [ ]:
customers['is_churned'] = (customers['status'] == 'inactive').astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Industry
industry_churn = customers.groupby('industry')['is_churned'].mean().sort_values(ascending=False)
sns.barplot(x=industry_churn.values, y=industry_churn.index, ax=axes[0], palette='Reds_r')
axes[0].set_title('Churn Rate by Industry')
axes[0].set_xlabel('Churn Rate')

# Country
country_churn = customers.groupby('country')['is_churned'].mean().sort_values(ascending=False).head(15)
sns.barplot(x=country_churn.values, y=country_churn.index, ax=axes[1], palette='Reds_r')
axes[1].set_title('Top 15 Countries by Churn Rate')
axes[1].set_xlabel('Churn Rate')

# Company Size
customers['size_group'] = pd.cut(customers['company_size'], 
                                  bins=[0, 10, 50, 100, 500, 10000],
                                  labels=['1-10', '11-50', '51-100', '101-500', '500+'])
size_churn = customers.groupby('size_group', observed=True)['is_churned'].mean()
sns.barplot(x=size_churn.index, y=size_churn.values, ax=axes[2], palette='Reds_r')
axes[2].set_title('Churn Rate by Company Size')
axes[2].set_xlabel('Company Size')
axes[2].set_ylabel('Churn Rate')

plt.tight_layout()
plt.show()

In [ ]:
# Company size distribution by churn status
fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(121)
sns.boxplot(x='status', y='company_size', data=customers, palette=['#2ecc71', '#e74c3c'], ax=ax1)
ax1.set_title('Company Size Distribution by Status')

ax2 = fig.add_subplot(122)
sns.histplot(data=customers, x='company_size', hue='status', kde=True, 
             palette={'active': '#2ecc71', 'inactive': '#e74c3c'}, alpha=0.6, ax=ax2)
ax2.set_title('Company Size Histogram by Status')

plt.tight_layout()
plt.show()

## 3. Subscription Analysis

In [ ]:
# Merge subscriptions with customers
cust_sub = customers.merge(subscriptions, on='customer_id', how='left', suffixes=('', '_sub'))
print(f'Merged shape: {cust_sub.shape}')
print(f'Customers without subscription: {cust_sub["plan_name"].isna().sum()}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plan distribution by churn
plan_churn = cust_sub.groupby('plan_name')['is_churned'].mean().sort_values(ascending=False)
sns.barplot(x=plan_churn.values, y=plan_churn.index, ax=axes[0], palette='Reds_r')
axes[0].set_title('Churn Rate by Plan')
axes[0].set_xlabel('Churn Rate')

# Monthly price distribution
sns.boxplot(x='status', y='monthly_price', data=cust_sub, palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_title('Monthly Price by Status')

# Monthly price vs churn
sns.kdeplot(data=cust_sub[cust_sub['status']=='active'], x='monthly_price', label='Active', color='#2ecc71', ax=axes[2])
sns.kdeplot(data=cust_sub[cust_sub['status']=='inactive'], x='monthly_price', label='Churned', color='#e74c3c', ax=axes[2])
axes[2].set_title('Monthly Price Distribution')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Subscription renewal date analysis
cust_sub['renewal_date'] = pd.to_datetime(cust_sub['renewal_date'])
cust_sub['days_to_renewal'] = (cust_sub['renewal_date'] - pd.Timestamp.now()).dt.days

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='status', y='days_to_renewal', data=cust_sub, palette=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Days to Renewal by Status')

sns.histplot(data=cust_sub, x='days_to_renewal', hue='status', kde=True,
             palette={'active': '#2ecc71', 'inactive': '#e74c3c'}, alpha=0.6, ax=axes[1])
axes[1].set_title('Days to Renewal Distribution')

plt.tight_layout()
plt.show()

## 4. Subscription Events Analysis

In [ ]:
# Analyze subscription events
event_counts = sub_events.groupby('customer_id').agg(
    total_events=('event_type', 'count'),
    upgrades=('event_type', lambda x: (x == 'upgrade').sum()),
    downgrades=('event_type', lambda x: (x == 'downgrade').sum())
).reset_index()

cust_events = customers.merge(event_counts, on='customer_id', how='left')
cust_events['total_events'] = cust_events['total_events'].fillna(0)
cust_events['upgrades'] = cust_events['upgrades'].fillna(0)
cust_events['downgrades'] = cust_events['downgrades'].fillna(0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(x='status', y='total_events', data=cust_events, palette=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Total Subscription Events by Status')

sns.boxplot(x='status', y='downgrades', data=cust_events, palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_title('Downgrades by Status')

sns.boxplot(x='status', y='upgrades', data=cust_events, palette=['#2ecc71', '#e74c3c'], ax=axes[2])
axes[2].set_title('Upgrades by Status')

plt.tight_layout()
plt.show()

## 5. Support Tickets Analysis

In [ ]:
# Aggregate ticket data per customer
ticket_agg = tickets.groupby('customer_id').agg(
    ticket_count=('id', 'count'),
    high_priority=('priority', lambda x: (x == 'High').sum()),
    open_tickets=('status', lambda x: (x != 'Closed').sum()),
    tech_issues=('category', lambda x: (x == 'Technical Issue').sum()),
    billing_issues=('category', lambda x: (x == 'Billing').sum())
).reset_index()

# Time to resolution
tickets['created_at'] = pd.to_datetime(tickets['created_at'])
tickets['resolved_at'] = pd.to_datetime(tickets['resolved_at'])
tickets['resolution_hours'] = (tickets['resolved_at'] - tickets['created_at']).dt.total_seconds() / 3600
tickets['resolution_hours'] = tickets['resolution_hours'].fillna(np.nan)

resolution_agg = tickets.groupby('customer_id')['resolution_hours'].mean().reset_index()
resolution_agg.columns = ['customer_id', 'avg_resolution_hours']

cust_tickets = customers.merge(ticket_agg, on='customer_id', how='left').merge(resolution_agg, on='customer_id', how='left')
ticket_cols = ['ticket_count', 'high_priority', 'open_tickets', 'tech_issues', 'billing_issues', 'avg_resolution_hours']
for c in ticket_cols:
    cust_tickets[c] = cust_tickets[c].fillna(0)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, col in enumerate(ticket_cols):
    ax = axes[idx//3][idx%3]
    sns.boxplot(x='status', y=col, data=cust_tickets, palette=['#2ecc71', '#e74c3c'], ax=ax)
    ax.set_title(f'{col.replace("_", " ").title()} by Status')

plt.tight_layout()
plt.show()

In [ ]:
# Ticket category distribution
fig = make_subplots(rows=1, cols=2, subplot_titles=['Ticket Categories', 'Ticket Priority Distribution'],
                    specs=[[{'type':'pie'}, {'type':'bar'}]], horizontal_spacing=0.2)

cat_counts = tickets['category'].value_counts()
fig.add_trace(go.Pie(labels=cat_counts.index, values=cat_counts.values, hole=0.4), row=1, col=1)

priority_counts = tickets.groupby(['category', 'priority']).size().reset_index(name='count')
fig.add_trace(go.Bar(x=priority_counts['category'], y=priority_counts['count'],
                     color=priority_counts['priority'], text=priority_counts['count']), row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

## 6. Activity Metrics Analysis

In [ ]:
# Aggregate activity metrics per customer
activity_agg = activity.groupby('customer_id').agg(
    avg_login_count=('login_count', 'mean'),
    total_login_count=('login_count', 'sum'),
    avg_session_duration=('session_duration', 'mean'),
    total_session_duration=('session_duration', 'sum'),
    avg_feature_usage=('feature_usage_score', 'mean'),
    max_feature_usage=('feature_usage_score', 'max'),
    avg_engagement=('engagement_score', 'mean'),
    max_engagement=('engagement_score', 'max'),
    avg_active_days=('active_days', 'mean'),
    total_active_days=('active_days', 'sum'),
    metric_count=('metric_date', 'count'),
    last_login_date=('metric_date', 'max')
).reset_index()

cust_activity = customers.merge(activity_agg, on='customer_id', how='left')

activity_features = ['avg_login_count', 'avg_session_duration', 'avg_feature_usage', 
                     'avg_engagement', 'avg_active_days', 'metric_count']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, col in enumerate(activity_features):
    ax = axes[idx//3][idx%3]
    sns.boxplot(x='status', y=col, data=cust_activity, palette=['#2ecc71', '#e74c3c'], ax=ax)
    ax.set_title(f'{col.replace("_", " ").title()} by Status')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of activity metrics
activity_corr = cust_activity.select_dtypes(include=[np.number]).corr()
plt.figure(figsize=(14, 10))
sns.heatmap(activity_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix of Customer Features')
plt.tight_layout()
plt.show()

## 7. Temporal Analysis

In [ ]:
# Customer registration trends
customers['created_at'] = pd.to_datetime(customers['created_at'])
customers['registration_month'] = customers['created_at'].dt.to_period('M').astype(str)

monthly_churn = customers.groupby('registration_month')['is_churned'].mean().reset_index()
monthly_counts = customers.groupby('registration_month').size().reset_index(name='count')

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(x=monthly_counts['registration_month'], y=monthly_counts['count'], 
                     name='New Customers', marker_color='lightblue'), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly_churn['registration_month'], y=monthly_churn['is_churned']*100,
                         name='Churn Rate %', marker_color='red', mode='lines+markers'), secondary_y=True)
fig.update_layout(title='Customer Registrations and Churn Rate Over Time', height=400)
fig.update_xaxes(title_text='Month')
fig.update_yaxes(title_text='New Customers', secondary_y=False)
fig.update_yaxes(title_text='Churn Rate %', secondary_y=True)
fig.show()

## 8. Key Insights Summary

In [ ]:
print("=" * 60)
print(" KEY EDA INSIGHTS")
print("=" * 60)

print(f"\n1. Overall churn rate: {customers['is_churned'].mean()*100:.2f}%")
print(f"2. Total customers: {len(customers)}, Churned: {customers['is_churned'].sum()}, Active: {(1-customers['is_churned']).sum()}")

print("\n3. Top industries with highest churn:")
for ind, rate in industry_churn.head(5).items():
    print(f"   - {ind}: {rate*100:.1f}%")

print("\n4. Subscription plans churn rates:")
for plan, rate in plan_churn.items():
    print(f"   - {plan}: {rate*100:.1f}%")

print("\n5. Key activity differences:")
for col in ['avg_login_count', 'avg_session_duration', 'avg_feature_usage', 'avg_engagement']:
    active_mean = cust_activity[cust_activity['status']=='active'][col].mean()
    churned_mean = cust_activity[cust_activity['status']=='inactive'][col].mean()
    diff_pct = ((churned_mean - active_mean) / active_mean) * 100
    print(f"   - {col}: Active={active_mean:.2f}, Churned={churned_mean:.2f} ({diff_pct:+.1f}% diff)")

print("\n6. Support tickets:")
active_tickets = cust_tickets[cust_tickets['status']=='active']['ticket_count'].mean()
churned_tickets = cust_tickets[cust_tickets['status']=='inactive']['ticket_count'].mean()
print(f"   - Avg tickets (Active): {active_tickets:.2f}")
print(f"   - Avg tickets (Churned): {churned_tickets:.2f}")

print("\n" + "=" * 60)
print("✅ EDA Complete - Ready for Feature Engineering")
print("=" * 60)